In [1]:
import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = "../the_execution_node/data/backtest_5m_3yr.parquet"
MODEL_PATH = "../the_models/meta_labeler_v2.json"
UNIVERSE_PATH = "../the_models/curated_universe.json"

# Load Universe
with open(UNIVERSE_PATH, "r") as f:
    baskets = json.load(f).get("baskets", {})

# Load Brain
meta_labeler = xgb.Booster()
meta_labeler.load_model(MODEL_PATH)

# Load Matrix
df = pd.read_parquet(DATA_PATH)
if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index)
df = df.sort_index().ffill().dropna()

print(f"[SUCCESS] Lab Ready: {df.shape[0]} rows loaded.")

[SUCCESS] Lab Ready: 315440 rows loaded.


In [ ]:
def simulate_realistic_strategy(z_thresh=2.0, ai_thresh=0.55, pt_skew=1.5, sl_skew=1.0, leverage=20.0):
    portfolio_returns = pd.Series(0.0, index=df.index)
    trades_executed = 0
    WARMUP_BARS = 2340 
    
    for spread_name, params in baskets.items():
        weights = params.get('weights', {})
        half_life = params.get('half_life', 1.0)
        allocation = params.get('capital_allocation', 0.0)
        
        if allocation <= 0: continue
            
        spread_val = pd.Series(0.0, index=df.index)
        for ticker, w in weights.items():
            col = f'{ticker}_price' if f'{ticker}_price' in df.columns else ticker
            if col in df.columns:
                spread_val += df[col] * w
                
        if spread_val.eq(0).all(): continue
                
        window = max(int(half_life * 78), 30)
        rolling_mean = spread_val.ewm(span=window).mean()
        rolling_std = spread_val.ewm(span=window).std().replace(0, np.nan)
        z_score = (spread_val - rolling_mean) / rolling_std
        
        base_signals = pd.Series(0, index=df.index)
        base_signals[z_score < -z_thresh] = 1
        base_signals[z_score > z_thresh] = -1
        
        price_change = spread_val.diff()
        tick_dir = np.sign(price_change).replace(0, np.nan).ffill().fillna(1)
        signed_vol = 100 * tick_dir
        covar = price_change.rolling(30).cov(price_change.shift(1))
        roll_measure = 2 * np.sqrt(np.abs(covar))
        
        features = pd.DataFrame({
            'frac_diff': price_change, 'volatility': rolling_std,
            'signal_strength': z_score, 'order_flow_imbalance': signed_vol.rolling(50).sum(),
            'effective_spread': roll_measure
        }).dropna()[['frac_diff', 'volatility', 'signal_strength', 'order_flow_imbalance', 'effective_spread']]
        
        valid_idx = features.index
        dmatrix = xgb.DMatrix(features)
        win_probs = meta_labeler.predict(dmatrix)
        
        meta_mask = pd.Series(win_probs > ai_thresh, index=valid_idx)
        final_signals = pd.Series(0, index=df.index)
        final_signals.loc[valid_idx] = base_signals.loc[valid_idx].where(meta_mask, 0)
        
        vol = spread_val.pct_change().ewm(span=100).std().fillna(0) * np.sqrt(78) 
        in_position, entry_price, entry_idx, current_vol = 0, 0.0, 0, 0.0
        trade_size_multiplier = 0.0
        
        for i in range(1, len(df)):
            current_price = spread_val.iloc[i]
            prev_price = spread_val.iloc[i-1]
            signal = final_signals.iloc[i]
            current_z = z_score.iloc[i]
            current_time = df.index[i].time()
            
            # 1. Entry Logic (With Warmup and Time Filter)
            if in_position == 0 and i > WARMUP_BARS:
                is_safe_time = (current_time >= pd.Timestamp("09:45").time()) and \
                               (current_time <= pd.Timestamp("15:45").time())
                
                if signal != 0 and is_safe_time:
                    in_position = signal
                    entry_price = current_price
                    entry_idx = i
                    current_vol = vol.iloc[i] if vol.iloc[i] > 0 else 0.005
                    trades_executed += 1
                    
                    # Half-Kelly Bet Sizing
                    prob = win_probs[i] if i < len(win_probs) else 0.55
                    kelly_fraction = prob - ((1.0 - prob) / 1.5)
                    trade_size_multiplier = max(0.0, kelly_fraction / 2.0)
                    
                    # Apply Brutal Entry Slippage
                    active_capital = allocation * trade_size_multiplier * leverage
                    entry_friction_cost = active_capital * 0.0005
                    portfolio_returns.iloc[i] -= entry_friction_cost
            
            # 2. Open Position Management
            elif in_position != 0:
                bars_held = i - entry_idx
                active_capital = allocation * trade_size_multiplier * leverage
                
                if prev_price != 0:
                    tick_ret = (current_price - prev_price) * in_position * active_capital
                    portfolio_returns.iloc[i] += tick_ret
                    
                trade_return = (current_price / entry_price - 1) * in_position if entry_price != 0 else 0
                
                # Triple Barrier & Mean Reversion Exit
                hit_pt = trade_return >= (current_vol * pt_skew)
                hit_sl = trade_return <= -(current_vol * sl_skew)
                hit_time = bars_held >= 120 
                hit_mean_rev = (in_position == 1 and current_z >= 0) or (in_position == -1 and current_z <= 0)
                
                if hit_pt or hit_sl or hit_time or hit_mean_rev:
                    # Apply Brutal Exit Slippage
                    exit_friction_cost = active_capital * 0.0005
                    portfolio_returns.iloc[i] -= exit_friction_cost
                    
                    in_position, entry_price, entry_idx, trade_size_multiplier = 0, 0.0, 0, 0.0

    # Slice off warmup to accurately calculate drawdown
    active_portfolio = portfolio_returns.iloc[WARMUP_BARS:]
    cum_returns = active_portfolio.cumsum()
    max_dd = (cum_returns - cum_returns.cummax()).min()
    total_pl = cum_returns.iloc[-1]
    
    return total_pl, max_dd, trades_executed

In [ ]:
import random

num_tests = 50
realistic_results = []

print(f"Running REALISTIC Monte Carlo Search ({num_tests} iterations)...\n")
print("This simulation includes 5bps slippage, Half-Kelly sizing, and Time Filters.\n")

for i in range(num_tests):
    # Randomize parameters to find the sweet spot that survives the friction
    test_z = round(random.uniform(1.8, 2.5), 2)     
    test_ai = round(random.uniform(0.55, 0.68), 2)  
    test_pt = round(random.uniform(1.0, 2.0), 2)    
    test_sl = round(random.uniform(1.0, 2.0), 2)    
    test_lev = round(random.uniform(10.0, 40.0), 1) 
    
    pl, dd, trades = simulate_realistic_strategy(
        z_thresh=test_z, ai_thresh=test_ai, 
        pt_skew=test_pt, sl_skew=test_sl, leverage=test_lev
    )
    
    romd = round(abs(pl / dd), 2) if dd < 0 else (999.99 if pl > 0 else 0)
    
    realistic_results.append({
        "AI_Conf": test_ai,
        "Z_Score": test_z,
        "PT_Skew": test_pt,
        "SL_Skew": test_sl,
        "Leverage": test_lev,
        "Total P&L": round(pl, 2),
        "Max Drawdown": round(dd, 2),
        "Trades": trades,
        "RoMD": romd
    })

# Display the Top 10 Configurations that survived the friction
realistic_df = pd.DataFrame(realistic_results).sort_values(by="RoMD", ascending=False)
print("=== TOP 10 REALISTIC CONFIGURATIONS (RANKED BY RoMD) ===")
realistic_df.head(10)

Running AFML Parameter Sweep...

Finished: Baseline (Current)
Finished: Strict AI, High Lev
Finished: Positive Skew (Let Winners Run)
Finished: Extreme Z-Score Sniping
Finished: The Institutional Mix


,Configuration,Total P&L,Max Drawdown,Trades,RoMD
4,The Institutional Mix,267.28,-1034.71,1779,0.26
0,Baseline (Current),201.16,-316.39,7150,0.64
1,"Strict AI, High Lev",90.23,-14.13,775,6.39
3,Extreme Z-Score Sniping,66.44,-1621.91,5669,0.04
2,Positive Skew (Let Winners Run),-10.31,-728.20,5856,0.01


In [4]:
import random

num_tests = 50
fine_tune_results = []

print(f"Running Fine-Grained Monte Carlo Search ({num_tests} iterations)...\n")

for i in range(num_tests):
    # Create microscopic variations around the winning parameters
    test_z = round(random.uniform(1.8, 2.4), 2)     # Baseline was 2.0
    test_ai = round(random.uniform(0.60, 0.72), 2)  # Baseline was 0.65
    test_pt = round(random.uniform(0.8, 1.4), 2)    # Baseline was 1.0
    test_sl = round(random.uniform(1.5, 3.0), 2)    # Baseline was 2.0
    test_lev = 20.0 # Lock leverage to cleanly compare pure mathematical edge
    
    pl, dd, trades = simulate_strategy(
        z_thresh=test_z, ai_thresh=test_ai, 
        pt_skew=test_pt, sl_skew=test_sl, leverage=test_lev
    )
    
    # Calculate RoMD securely
    romd = round(abs(pl / dd), 2) if dd < 0 else (999.99 if pl > 0 else 0)
    
    fine_tune_results.append({
        "AI_Conf": test_ai,
        "Z_Score": test_z,
        "PT_Skew": test_pt,
        "SL_Skew": test_sl,
        "Total P&L": round(pl, 2),
        "Max Drawdown": round(dd, 2),
        "Trades": trades,
        "RoMD": romd
    })

# Display the Top 10 Absolute Best Configurations based strictly on RoMD
fine_tune_df = pd.DataFrame(fine_tune_results).sort_values(by="RoMD", ascending=False)
print("=== TOP 10 MONTE CARLO CONFIGURATIONS (RANKED BY RoMD) ===")
fine_tune_df.head(10)

Running Fine-Grained Monte Carlo Search (50 iterations)...



KeyboardInterrupt: 